# From Commodity Shock to Checkout Price
## Measuring Price Stickiness Between Global Commodity Markets and U.S. Grocery Prices

### Introduction
This final project dives into the real-world disconnect between global commodity markets and the actual prices consumers pay at U.S. grocery stores. The core motivation here is pretty straightforward: when the wholesale cost of staples like wheat, coffee, bananas, or chicken fluctuates, we rarely see the impact at the supermarket right away. Instead, retail prices tend to drag their feet, only pass on a fraction of the changes, or react totally differently depending on whether raw costs are going up or down.

To map out exactly how this ripple effect plays out over time, this notebook pairs global commodity data from the World Bank with U.S. Bureau of Labor Statistics retail grocery stats. The setup is designed to serve as both a formal analytical report and a fully reproducible data workflow, using pandas to handle the data prep behind the scenes and altair to build out the interactive visualizations.


## Problem Context / Question

In economics, there's a concept known as "price stickiness," which is essentially just the tendency for prices to react sluggishly to changing market conditions instead of adjusting overnight. In the grocery world, this lag happens for a variety of very practical reasons—things like locked-in supplier contracts, shipping delays, backlogged inventory, processing costs, and calculated retail strategies. Because of all these moving parts, a massive shake-up in global wholesale markets might not be felt at the local supermarket until much later.

This project zeroes in on a specific two-part question:

Just how long does it take for shifts in global commodity prices to actually hit U.S. grocery shelves? And on top of that, do retailers raise prices faster when their costs go up than they lower them when costs go down?

To get to the bottom of this, the analysis tackles the data by:

Tracking and comparing the historical timelines of raw material costs versus everyday retail prices.

Measuring the actual gaps between upstream commodity costs and downstream price tags.

Looking at month-to-month volatility to spot exactly when and where retail prices dig their heels in and get "sticky."

In [4]:

%pip install -q altair


In [5]:

import pandas as pd
import altair as alt


# we disable the limit so that the full merged dataset can be visualized.
alt.data_transformers.disable_max_rows()

# Define the exact filenames used in this notebook.
# If the files are uploaded to Colab under different names, only these two
# variables need to be edited rather than the full workflow.
WORLD_BANK_FILE = "wb_commodity_price_intelligence_1960_2026.csv"
BLS_FILE = "bls_retail_grocieries.csv"

# Improve notebook display defaults so wide DataFrames are easier to inspect.
pd.set_option("display.max_columns", 60)


In [6]:

# Load and clean the World Bank commodity dataset

wb = pd.read_csv(WORLD_BANK_FILE)

# Keep only the columns required for this project and rename them to a common schema.
# Using a shared schema across datasets makes the later merge logic clearer.
wb = (
    wb.rename(
        columns={
            "date": "Date",
            "commodity_name": "Commodity",
            "price_nominal_usd": "Price",
        }
    )[["Date", "Commodity", "Price"]]
    .copy()
)

# Convert the date strings into pandas datetime objects.
# Any unparsable values become missing values and are removed later.
wb["Date"] = pd.to_datetime(wb["Date"], errors="coerce")

# Convert the price column to numeric form in case the source contains non-numeric text.
wb["Price"] = pd.to_numeric(wb["Price"], errors="coerce")

# Drop incomplete rows because the analysis requires valid dates, product labels,
# and price values for every record that enters the merge stage.
wb = wb.dropna(subset=["Date", "Commodity", "Price"]).copy()

# Standardize the date to the first day of the month.
# This ensures consistent monthly alignment across both datasets.
wb["Date"] = wb["Date"].dt.to_period("M").dt.to_timestamp()

# Display a small preview so the file structure can be confirmed in Colab.
print("World Bank commodity data preview:")
display(wb.head())

# Show a few available commodity labels to make the matching logic transparent.
print("Sample World Bank commodity names:")
display(pd.DataFrame(sorted(wb["Commodity"].unique())[:25], columns=["Commodity"]))


World Bank commodity data preview:


,Date,Commodity,Price
0,1960-01-01,Aluminum,511.471832
1,1960-02-01,Aluminum,511.471832
2,1960-03-01,Aluminum,511.471832
3,1960-04-01,Aluminum,511.471832
4,1960-05-01,Aluminum,511.471832


Sample World Bank commodity names:


,Commodity
0,Aluminum
1,"Banana, Europe"
2,"Banana, US"
3,Barley
4,Beef
5,Chicken
6,"Coal, Australian"
7,"Coal, South African"
8,Cocoa
9,Coconut oil


In [7]:

# Parse and tidy the BLS retail grocery dataset


def parse_bls_block_file(path):
    # Read the full file as plain text lines so we can process the block layout manually.
    with open(path, "r", encoding="utf-8-sig") as file_handle:
        lines = file_handle.read().splitlines()

    # Define the month labels in the exact order used by the BLS file.
    month_names = [
        "Jan", "Feb", "Mar", "Apr", "May", "Jun",
        "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
    ]

    # This list will collect one tidy monthly record at a time.
    records = []
    line_index = 0

    # Walk through the file line by line, identifying each item block.
    while line_index < len(lines):
        current_line = lines[line_index].strip()

        # A new item block always starts with a Series Id line.
        if current_line.startswith("Series Id:"):
            # Read the small metadata section attached to the item.
            series_id = current_line.split(":", 1)[1].strip()
            series_title = lines[line_index + 1].split(":", 1)[1].strip() if line_index + 1 < len(lines) else None
            area = lines[line_index + 2].split(":", 1)[1].strip() if line_index + 2 < len(lines) else None
            item = lines[line_index + 3].split(":", 1)[1].strip() if line_index + 3 < len(lines) else None

            # Move forward until the monthly matrix header appears.
            data_index = line_index + 4
            while data_index < len(lines) and not lines[data_index].startswith("Year,"):
                data_index += 1

            # If the month header is missing, skip this malformed block safely.
            if data_index >= len(lines):
                line_index += 1
                continue

            # Advance past the header row so parsing begins on the yearly records.
            data_index += 1

            # Read each year row until a new block begins or the file ends.
            while data_index < len(lines):
                row_text = lines[data_index].strip()

                # Skip empty lines inside the block.
                if not row_text:
                    data_index += 1
                    continue

                # A new Series Id marks the beginning of the next grocery item.
                if row_text.startswith("Series Id:"):
                    break

                # Split the comma-delimited monthly row into fields.
                row_parts = [part.strip() for part in row_text.split(",")]

                # Valid rows contain one year plus 12 monthly values.
                if len(row_parts) == 13 and row_parts[0].isdigit():
                    year_value = int(row_parts[0])

                    # Create one tidy monthly record for each non-missing month.
                    for month_number, month_name in enumerate(month_names, start=1):
                        price_text = row_parts[month_number]

                        # Ignore blank or placeholder values instead of forcing errors.
                        if price_text in ("", "-", "NA"):
                            continue

                        records.append(
                            {
                                "Series_Id": series_id,
                                "Series_Title": series_title,
                                "Area": area,
                                "Commodity": item,
                                "Year": year_value,
                                "Month_Label": month_name,
                                "Date": pd.Timestamp(year=year_value, month=month_number, day=1),
                                "Price": pd.to_numeric(price_text, errors="coerce"),
                            }
                        )

                data_index += 1

            # Continue scanning from the next block rather than from the old header line.
            line_index = data_index
        else:
            line_index += 1

    # Convert the collected monthly records into a tidy pandas DataFrame.
    bls_tidy = pd.DataFrame(records)

    # Remove incomplete rows that cannot be analyzed or merged.
    bls_tidy = bls_tidy.dropna(subset=["Date", "Commodity", "Price"]).copy()

    # Standardize to monthly timestamps for consistent joins.
    bls_tidy["Date"] = bls_tidy["Date"].dt.to_period("M").dt.to_timestamp()

    return bls_tidy

# Parse the uploaded BLS file into a long-format DataFrame.
bls = parse_bls_block_file(BLS_FILE)

# Display a preview so the user can confirm that the block parser worked.
print("BLS retail grocery data preview:")
display(bls.head())

# Show the distinct grocery item names that are available for pairing.
print("Available BLS grocery items:")
display(pd.DataFrame(sorted(bls["Commodity"].unique()), columns=["Commodity"]))


BLS retail grocery data preview:


,Series_Id,Series_Title,Area,Commodity,Year,Month_Label,Date,Price
0,APU0000704111,"Bacon, sliced, per lb. (453.6 gm) in U.S. city...",U.S. city average,"Bacon, sliced, per lb. (453.6 gm)",1980,Jan,1980-01-01,1.453
1,APU0000704111,"Bacon, sliced, per lb. (453.6 gm) in U.S. city...",U.S. city average,"Bacon, sliced, per lb. (453.6 gm)",1980,Feb,1980-02-01,1.410
2,APU0000704111,"Bacon, sliced, per lb. (453.6 gm) in U.S. city...",U.S. city average,"Bacon, sliced, per lb. (453.6 gm)",1980,Mar,1980-03-01,1.361
3,APU0000704111,"Bacon, sliced, per lb. (453.6 gm) in U.S. city...",U.S. city average,"Bacon, sliced, per lb. (453.6 gm)",1980,Apr,1980-04-01,1.323
4,APU0000704111,"Bacon, sliced, per lb. (453.6 gm) in U.S. city...",U.S. city average,"Bacon, sliced, per lb. (453.6 gm)",1980,May,1980-05-01,1.266


Available BLS grocery items:


,Commodity
0,"Bacon, sliced, per lb. (453.6 gm)"
1,"Bananas, per lb. (453.6 gm)"
2,"Bread, white, pan, per lb. (453.6 gm)"
3,"Chicken, fresh, whole, per lb. (453.6 gm)"
4,"Coffee, 100%, ground roast, all sizes, per lb...."
5,"Eggs, grade A, large, per doz."
6,Electricity per KWH
7,"Flour, white, all purpose, per lb. (453.6 gm)"
8,Fuel oil #2 per gallon (3.785 liters)
9,"Gasoline, all types, per gallon/3.785 liters"


In [8]:

# Create commodity pair mappings and build the merged analytical dataset
# Each row below links one World Bank raw commodity to one BLS retail grocery item.
# The label in Commodity_Pair is the shared identifier used for comparisons,
# normalization, filtering, and visualization.
pair_definitions = pd.DataFrame(
    [
        {
            "Raw_Commodity": "Wheat, US HRW",
            "Retail_Item": "Bread, white, pan, per lb. (453.6 gm)",
            "Commodity_Pair": "Wheat vs. Bread",
        },
        {
            "Raw_Commodity": "Coffee, Arabica",
            "Retail_Item": "Coffee, 100%, ground roast, all sizes, per lb. (453.6 gm)",
            "Commodity_Pair": "Coffee Beans vs. Roasted Coffee",
        },
        {
            "Raw_Commodity": "Rice, Thai 5%",
            "Retail_Item": "Rice, white, long grain, uncooked, per lb. (453.6 gm)",
            "Commodity_Pair": "Rice vs. Retail Rice",
        },
        {
            "Raw_Commodity": "Banana, US",
            "Retail_Item": "Bananas, per lb. (453.6 gm)",
            "Commodity_Pair": "Bananas (Global) vs. Bananas (Retail)",
        },
        {
            "Raw_Commodity": "Orange",
            "Retail_Item": "Oranges, Navel, per lb. (453.6 gm)",
            "Commodity_Pair": "Orange Market vs. Retail Oranges",
        },
        {
            "Raw_Commodity": "Chicken",
            "Retail_Item": "Chicken, fresh, whole, per lb. (453.6 gm)",
            "Commodity_Pair": "Chicken Market vs. Whole Chicken",
        },
    ]
)

# Verify which pair definitions are actually present in the uploaded files.
# This makes the notebook more robust because missing labels can be identified clearly.
available_pairs = pair_definitions[
    pair_definitions["Raw_Commodity"].isin(wb["Commodity"])
    & pair_definitions["Retail_Item"].isin(bls["Commodity"])
].copy()

# Stop early if no valid overlap exists between the two datasets.
if available_pairs.empty:
    raise ValueError(
        "No valid commodity pairs were found. Check the uploaded filenames and item labels."
    )

print("Commodity pairs available for analysis:")
display(available_pairs)

# Keep only the World Bank rows that belong to one of the approved raw commodities.
wb_pairs = (
    wb.merge(
        available_pairs[["Raw_Commodity", "Commodity_Pair"]],
        left_on="Commodity",
        right_on="Raw_Commodity",
        how="inner",
    )
    .drop(columns=["Raw_Commodity"])
    .rename(columns={"Price": "Raw_Price", "Commodity": "Raw_Commodity_Name"})
)

# Keep only the BLS rows that belong to one of the approved retail grocery items.
bls_pairs = (
    bls.merge(
        available_pairs[["Retail_Item", "Commodity_Pair"]],
        left_on="Commodity",
        right_on="Retail_Item",
        how="inner",
    )
    .drop(columns=["Retail_Item"])
    .rename(columns={"Price": "Retail_Price", "Commodity": "Retail_Item_Name"})
)

# Aggregate both datasets to a single observation per month and commodity pair.
# Mean is used as a safe monthly aggregator in case duplicate rows exist after parsing.
wb_monthly = (
    wb_pairs.groupby(["Date", "Commodity_Pair"], as_index=False)["Raw_Price"]
    .mean()
)

bls_monthly = (
    bls_pairs.groupby(["Date", "Commodity_Pair"], as_index=False)["Retail_Price"]
    .mean()
)

# Merge the upstream and downstream monthly series on the exact keys required by the project:
# Date and Commodity_Pair.
merged = (
    wb_monthly.merge(bls_monthly, on=["Date", "Commodity_Pair"], how="inner")
    .sort_values(["Commodity_Pair", "Date"])
    .reset_index(drop=True)
)

# Build a reusable helper that normalizes any time series to a base-100 index.
# A value of 125 means the price is 25% above its first observed level in that pair.
def base_100_index(series):
    first_valid_value = series.dropna().iloc[0]
    return (series / first_valid_value) * 100

# Create normalized indexes for the raw and retail series separately within each pair.
merged["Raw_Index"] = merged.groupby("Commodity_Pair")["Raw_Price"].transform(base_100_index)
merged["Retail_Index"] = merged.groupby("Commodity_Pair")["Retail_Price"].transform(base_100_index)

# Compute a normalized margin measure.
# Because raw and retail prices are reported in different units, this project interprets
# the margin as the difference between the two normalized indexes rather than a literal
# dollar spread.
merged["Margin_Index_Points"] = merged["Retail_Index"] - merged["Raw_Index"]

# Compute month-over-month percentage changes for both sides of the supply chain.
# These features are used in the volatility and stickiness visualization.
merged["Raw_MoM_Pct"] = merged.groupby("Commodity_Pair")["Raw_Price"].pct_change() * 100
merged["Retail_MoM_Pct"] = merged.groupby("Commodity_Pair")["Retail_Price"].pct_change() * 100

# Create a preferred display order so the bread pair appears first when available.
preferred_order = [
    "Wheat vs. Bread",
    "Coffee Beans vs. Roasted Coffee",
    "Rice vs. Retail Rice",
    "Bananas (Global) vs. Bananas (Retail)",
    "Orange Market vs. Retail Oranges",
    "Chicken Market vs. Whole Chicken",
]
available_order = [pair for pair in preferred_order if pair in merged["Commodity_Pair"].unique()]

# Display a preview and a count of monthly observations for each pair.
print("Merged analytical dataset preview:")
display(merged.head())

print("Observations per commodity pair:")
display(merged["Commodity_Pair"].value_counts().rename_axis("Commodity_Pair").reset_index(name="Observations"))


Commodity pairs available for analysis:


,Raw_Commodity,Retail_Item,Commodity_Pair
0,"Wheat, US HRW","Bread, white, pan, per lb. (453.6 gm)",Wheat vs. Bread
1,"Coffee, Arabica","Coffee, 100%, ground roast, all sizes, per lb....",Coffee Beans vs. Roasted Coffee
2,"Rice, Thai 5%","Rice, white, long grain, uncooked, per lb. (45...",Rice vs. Retail Rice
3,"Banana, US","Bananas, per lb. (453.6 gm)",Bananas (Global) vs. Bananas (Retail)
4,Orange,"Oranges, Navel, per lb. (453.6 gm)",Orange Market vs. Retail Oranges
5,Chicken,"Chicken, fresh, whole, per lb. (453.6 gm)",Chicken Market vs. Whole Chicken


Merged analytical dataset preview:


,Date,Commodity_Pair,Raw_Price,Retail_Price,Raw_Index,Retail_Index,Margin_Index_Points,Raw_MoM_Pct,Retail_MoM_Pct
0,1980-01-01,Bananas (Global) vs. Bananas (Retail),0.4018,0.319,100.000000,100.000000,0.000000,NaN,NaN
1,1980-02-01,Bananas (Global) vs. Bananas (Retail),0.3722,0.345,92.633151,108.150470,15.517319,-7.366849,8.150470
2,1980-03-01,Bananas (Global) vs. Bananas (Retail),0.4227,0.355,105.201593,111.285266,6.083674,13.567974,2.898551
3,1980-04-01,Bananas (Global) vs. Bananas (Retail),0.3956,0.353,98.456944,110.658307,12.201363,-6.411166,-0.563380
4,1980-05-01,Bananas (Global) vs. Bananas (Retail),0.4442,0.357,110.552514,111.912226,1.359712,12.285137,1.133144


Observations per commodity pair:


,Commodity_Pair,Observations
0,Wheat vs. Bread,552
1,Chicken Market vs. Whole Chicken,539
2,Bananas (Global) vs. Bananas (Retail),538
3,Rice vs. Retail Rice,515
4,Coffee Beans vs. Roasted Coffee,500
5,Orange Market vs. Retail Oranges,396


## Visualization 1: The Delay Detector




In [9]:

# Visualization 1: The Delay Detector
# Select a default commodity pair for the detailed time-series comparison.
# The preferred order above makes the notebook narration more intuitive when the
# wheat-bread pair is available, while still remaining robust if a pair is missing.
pair_for_delay_detector = available_order[0] if available_order else merged["Commodity_Pair"].iloc[0]

# Filter the merged dataset down to the selected pair and keep only the fields
# needed for the two-line indexed time-series chart.
delay_df = merged.loc[
    merged["Commodity_Pair"] == pair_for_delay_detector,
    ["Date", "Raw_Index", "Retail_Index"]
].copy()

# Reshape the data from wide to long format so that Altair can encode the raw
# and retail series with a single color legend.
delay_long = delay_df.melt(
    id_vars="Date",
    value_vars=["Raw_Index", "Retail_Index"],
    var_name="Series",
    value_name="Index_Value"
)

# Replace technical field names with reader-friendly labels used in the chart legend.
delay_long["Series"] = delay_long["Series"].map(
    {
        "Raw_Index": "Raw commodity index",
        "Retail_Index": "Retail grocery index",
    }
)

# Create an interval brush bound to the x-axis only.
# This brush lives in the lower context chart and controls which dates appear
# in the main time-series view above it.
time_brush = alt.selection_interval(encodings=["x"])

# Build the main detail chart that updates when the brush selection changes.
delay_main = (
    alt.Chart(delay_long)
    .mark_line(point=True)
    .encode(
        x=alt.X("Date:T", title="Month"),
        y=alt.Y("Index_Value:Q", title="Normalized price index (base = 100)"),
        color=alt.Color("Series:N", title="Series"),
        tooltip=[
            alt.Tooltip("Date:T", title="Month"),
            alt.Tooltip("Series:N", title="Series"),
            alt.Tooltip("Index_Value:Q", title="Index value", format=".2f"),
        ],
    )
    .transform_filter(time_brush)
    .properties(
        width=800,
        height=360,
        title=f"Delay Detector: {pair_for_delay_detector}",
    )
)

# Build the smaller context chart that shows the full time span and hosts the brush.
delay_context = (
    alt.Chart(delay_long)
    .mark_line()
    .encode(
        x=alt.X("Date:T", title="Full timeline"),
        y=alt.Y("Index_Value:Q", title=None),
        color=alt.Color("Series:N", title="Series"),
    )
    .add_params(time_brush)
    .properties(width=800, height=90)
)

# Stack the context view under the detail view to create the overview-detail interaction.
delay_detector_chart = alt.vconcat(delay_main, delay_context).resolve_scale(color="shared")

# Render the final interactive chart.
delay_detector_chart


alt.VConcatChart(...)

## Visualization 2: The Margin Explorer


In [12]:
# -------------------------
# Visualization 2: The Margin Explorer
# -------------------------
# Reuse the same pair as Visualization 1 so that the notebook narrative builds
# from timing comparison into correlation and margin analysis.
pair_for_margin_explorer = pair_for_delay_detector

# Keep only the fields needed for the scatter plot and linked margin chart.
# The scatter uses actual raw and retail prices because the goal is to inspect
# correlation between observed upstream and downstream price levels.
margin_df = merged.loc[
    merged["Commodity_Pair"] == pair_for_margin_explorer,
    ["Date", "Raw_Price", "Retail_Price", "Margin_Index_Points"]
].copy()

# Create a two-dimensional interval selection on the scatter plot.
# Brushing this view will cross-filter the linked bar chart on the right.
scatter_brush = alt.selection_interval()

# Build the scatter plot using actual price levels.
# Opacity and color change dynamically so brushed observations stand out.
scatter = (
    alt.Chart(margin_df)
    .mark_circle(size=90)
    .encode(
        x=alt.X("Raw_Price:Q", title="Raw commodity price"),
        y=alt.Y("Retail_Price:Q", title="Retail grocery price"),
        color=alt.condition(scatter_brush, alt.value("#1f77b4"), alt.value("#c7c7c7")),
        opacity=alt.condition(scatter_brush, alt.value(0.9), alt.value(0.35)),
        tooltip=[
            alt.Tooltip("Date:T", title="Month"),
            alt.Tooltip("Raw_Price:Q", title="Raw price", format=".3f"),
            alt.Tooltip("Retail_Price:Q", title="Retail price", format=".3f"),
            alt.Tooltip("Margin_Index_Points:Q", title="Margin (index points)", format=".2f"),
        ],
    )
    .add_params(scatter_brush)
    .properties(
        width=360,
        height=320,
        title=f"Raw vs. Retail Correlation: {pair_for_margin_explorer}",
    )
)

# Create a zero reference line to anchor the margin interpretation.
zero_rule = (
    alt.Chart(pd.DataFrame({"y": [0]}))
    .mark_rule(color="black", strokeDash=[4, 4])
    .encode(y="y:Q")
)

# Build the linked bar chart of the normalized margin over time.
# The transform_filter call is the key interaction: only dates corresponding to
# the brushed scatter points remain visible.
margin_bars = (
    alt.Chart(margin_df)
    .mark_bar()
    .encode(
        x=alt.X("Date:T", title="Month"),
        y=alt.Y("Margin_Index_Points:Q", title="Retail minus raw (index points)"),
        color=alt.condition(
            alt.datum.Margin_Index_Points >= 0,
            alt.value("#d95f02"),
            alt.value("#1b9e77")
        ),
        tooltip=[
            alt.Tooltip("Date:T", title="Month"),
            alt.Tooltip("Margin_Index_Points:Q", title="Margin (index points)", format=".2f"),
        ],
    )
    .transform_filter(scatter_brush)
    .properties(width=420, height=320, title="Linked Margin Over Time")
)

# Place the two charts side by side so the cross-filtering is immediately visible.
margin_explorer_chart = alt.hconcat(scatter, zero_rule + margin_bars)

# Render the second interactive visualization.
margin_explorer_chart


alt.HConcatChart(...)

## Visualization 3: Volatility & Stickiness Viewer




In [11]:

# Reshape the two month-over-month series into a single tidy table so that Altair
# can draw both lines from one dataset and filter them with a dropdown parameter.
volatility_long = (
    merged.melt(
        id_vars=["Date", "Commodity_Pair"],
        value_vars=["Raw_MoM_Pct", "Retail_MoM_Pct"],
        var_name="Series",
        value_name="MoM_Change"
    )
    .dropna()
    .copy()
)

# Replace technical series names with reader-friendly labels for the legend.
volatility_long["Series"] = volatility_long["Series"].map(
    {
        "Raw_MoM_Pct": "Raw commodity MoM %",
        "Retail_MoM_Pct": "Retail grocery MoM %",
    }
)

# Create the dropdown options from the pairs that actually survived the merge step.
pair_options = available_order if available_order else sorted(volatility_long["Commodity_Pair"].unique())

# Create a dropdown-bound Altair parameter.
# Changing the selected value updates the chart instantly by filtering the data.
pair_dropdown = alt.binding_select(options=pair_options, name="Commodity Pair: ")
selected_pair = alt.param(name="selected_pair", bind=pair_dropdown, value=pair_options[0])

# Add a zero line so positive and negative monthly changes are easier to distinguish.
volatility_zero_rule = (
    alt.Chart(pd.DataFrame({"y": [0]}))
    .mark_rule(color="black", strokeDash=[4, 4])
    .encode(y="y:Q")
)

# Build the dropdown-driven line chart.
# The transform_filter step references the selected_pair parameter to keep only
# the currently chosen commodity pair visible.
volatility_lines = (
    alt.Chart(volatility_long)
    .mark_line(point=True)
    .encode(
        x=alt.X("Date:T", title="Month"),
        y=alt.Y("MoM_Change:Q", title="Month-over-month percentage change"),
        color=alt.Color("Series:N", title="Series"),
        tooltip=[
            alt.Tooltip("Commodity_Pair:N", title="Commodity pair"),
            alt.Tooltip("Date:T", title="Month"),
            alt.Tooltip("Series:N", title="Series"),
            alt.Tooltip("MoM_Change:Q", title="MoM change (%)", format=".2f"),
        ],
    )
    .add_params(selected_pair)
    .transform_filter(alt.datum.Commodity_Pair == selected_pair)
    .properties(width=820, height=360, title="Volatility & Stickiness Viewer")
)

# Layer the zero line under the volatility lines and render the final interactive chart.
volatility_viewer_chart = volatility_zero_rule + volatility_lines
volatility_viewer_chart


alt.LayerChart(...)

## Unexpected Learnings

One of the more surprising takeaways from this project is how retail prices for staples—like bread or coffee—can stay stubbornly high long after global commodity prices start to drop. You’d naturally expect a cheaper global wheat or coffee market to quickly translate into savings at the grocery store. But looking at the charts, a totally different story emerges: wholesale costs drop off rapidly, while checkout prices just plateau or drift down painfully slowly.

This goes to show that raw commodity costs are only one piece of the puzzle when it comes to predicting grocery bills. The lag we see in the data actually reflects everything else that goes into getting food on the shelf, like processing, packaging, shipping, labor, and long-term inventory contracts. It also points to a frustrating but common retail reality: prices are usually quick to go up but slow to come down. Ultimately, it’s a great reminder that consumer food inflation is dictated by the entire supply chain, not just what's happening in the global commodity markets.


## Concluding Comments

This project uses interactive Altair visualizations to explore a familiar question: when global commodity prices shift, do U.S. grocery prices react right away? The short answer is no. Even though raw material costs and retail prices usually trend in the same general direction, the timing and intensity rarely line up perfectly.

The analysis is broken down into three main views. First, the Delay Detector highlights the lag between raw costs shifting and those changes actually hitting the shelves. Next, the Margin Explorer points out specific periods where retail prices completely decoupled from raw commodity trends. Finally, the Volatility & Stickiness Viewer illustrates how commodity markets bounce around significantly more than stable—but stubborn—grocery prices.

The main takeaway is that retail food prices are inherently "sticky." They take their time adjusting downward and frequently stay high well after global commodity prices have already started to drop. Ultimately, the data points to supply chain bottlenecks, corporate pricing strategies, and overall market structure as the real reasons why consumers rarely see immediate relief at the checkout line when global markets improve.
